In [1]:
import pandas as pd

In [24]:
"""
resolve_foodex2_facets.py
─────────────────────────
Enriches every term in foodHierarchy_nlp.csv with human-readable facet
descriptions resolved from MTX_17.0.csv.

INPUT
    foodHierarchy_nlp.csv  →  FOOD hierarchy (termCode, termExtendedName, allFacets, …)
    MTX_17.0.csv           →  Full MTX catalogue — ALL hierarchies incl. facets

OUTPUT
    foodHierarchy_nlp_extended.csv  →  original + two new columns:
        • facets_resolved   : JSON string  [{facet_cat, code, name}, …]
        • termExtendedName_full : "base term, source: X, part: Y, process: Z, …"

USAGE (Colab)
    Adjust FOOD_CSV / MTX_CSV paths below → Run All.
"""

import re
import json
import pandas as pd
from pathlib import Path

# ═══════════════════════════════════════════════════════════════════════════════
# 0.  PATHS — adjust to your Google Drive layout
# ═══════════════════════════════════════════════════════════════════════════════


FOOD_CSV   = "/home/alexl/TFM/eda/foodHierarchy.xlsx"
MTX_CSV    = "/home/alexl/TFM/data/MTX_17.0.xlsx"
OUTPUT_CSV = "/home/alexl/TFM/data/foodHierarchy_nlp_extended.csv"





In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1.  LOAD & INSPECT
# ═══════════════════════════════════════════════════════════════════════════════
food_df = pd.read_excel(FOOD_CSV)
print(f'FOOD hierarchy : {len(food_df):,} rows')
print(f'  Columns      : {list(food_df.columns)}')
print(f'  allFacets NaN: {food_df["allFacets"].isna().sum():,} / {len(food_df):,}')
print()

In [11]:
mtx_df = pd.read_excel(MTX_CSV, sheet_name='term')
print(f'MTX catalogue  : {len(mtx_df):,} rows')
print(f'  Columns      : {list(mtx_df.columns)}')
print()

MTX catalogue  : 31,680 rows
  Columns      : ['termCode', 'termExtendedName', 'termShortName', 'termScopeNote', 'version', 'lastUpdate', 'validFrom', 'validTo', 'status', 'deprecated', 'scientificNames', 'commonNames', 'allFacets', 'ISSCAAP', 'taxonomicCode', 'alpha3Code', 'GEMSCode', 'matrixCode', 'LangualCode', 'foodexOldCode', 'implicitFacets', 'detailLevel', 'termType', 'prodTreat', 'prodMeth', 'prodPack', 'EuringsCode', 'IFNCode', 'EUFeedReg', 'EPPOCode', 'VectorNetCode', 'ADDFOODCode', 'masterFlag', 'masterParentCode', 'masterOrder', 'masterReportable', 'masterHierarchyCode', 'reportFlag', 'reportParentCode', 'reportOrder', 'reportReportable', 'reportHierarchyCode', 'pestFlag', 'pestParentCode', 'pestOrder', 'pestReportable', 'pestHierarchyCode', 'biomoFlag', 'biomoParentCode', 'biomoOrder', 'biomoReportable', 'biomoHierarchyCode', 'feedFlag', 'feedParentCode', 'feedOrder', 'feedReportable', 'feedHierarchyCode', 'expoFlag', 'expoParentCode', 'expoOrder', 'expoReportable', 'expoH

In [ ]:
mtx_name_col = 'termExtendedName'
# ── Build code → name lookup from MTX ──────────────────────────────────────
code_to_name = (
    mtx_df.dropna(subset=['termCode', mtx_name_col])
    .drop_duplicates(subset='termCode', keep='first')
    .set_index('termCode')[mtx_name_col]
    .to_dict()
)
print(f'Lookup table   : {len(code_to_name):,} unique codes')


Lookup table   : 31,680 unique codes


In [19]:
code_to_name

{'A000A': 'Teff grain',
 'A000B': 'Finger millet grain',
 'A000C': 'African millet grain',
 'A000D': 'Foxtail millet grain',
 'A000E': 'Little millet grain',
 'A000F': 'Oat and similar-',
 'A000G': 'Oat grain',
 'A000H': 'Oat grain, red',
 'A000J': 'Grains and grain-based products',
 'A000K': 'Cereals and cereal primary derivatives',
 'A000L': 'Cereal grains (and cereal-like grains)',
 'A000M': 'Amaranth grains',
 'A000N': 'Buckwheat',
 'A000P': 'Barley grains',
 'A000Q': 'Kaniwa grain',
 'A000R': 'Quinoa grain',
 'A000S': 'Maize and similar-',
 'A000T': 'Maize grain',
 'A000V': 'Popcorn kernels',
 'A000X': 'Teosinte grain',
 'A000Y': 'Common millet and similar-',
 'A000Z': 'Barnyard millet',
 'A001A': 'Pearl millet grain',
 'A001B': 'Common millet grain',
 'A001C': 'Rice and similar-',
 'A001D': 'Rice grain',
 'A001E': 'Rice grain, brown',
 'A001F': 'Rice grain, long-grain',
 'A001G': 'Rice grain, mixed',
 'A001H': 'Rice grain, red',
 'A001J': 'Indian rice grain',
 'A001K': 'Rye grain

In [14]:
# ═══════════════════════════════════════════════════════════════════════════════
# 2.  FACET CATEGORY LABELS
# ═══════════════════════════════════════════════════════════════════════════════
# Official FoodEx2 facet categories (MTX 2.1 / EFSA catalogue browser)
FACET_LABELS = {
    'F01': 'source',
    'F02': 'part-nature',
    'F03': 'process',
    'F04': 'packaging',
    'F05': 'cooking',
    'F06': 'production',
    'F07': 'target-consumer',
    'F08': 'qualification',
    'F09': 'component',
    'F10': 'contact-material',
    'F11': 'additive',
    'F12': 'pesticide',
    'F13': 'veterinary-drug',
    'F14': 'flavouring',
    'F15': 'fortification',
    'F16': 'genetically-modified',
    'F17': 'allergen',
    'F18': 'contaminant',
    'F19': 'brand',
    'F20': 'physical-state',
    'F21': 'colour',
    'F22': 'shape',
    'F23': 'fat-content',
    'F24': 'caffeine-content',
    'F25': 'species',
    'F26': 'maturity',
    'F27': 'geographical-origin',
    'F28': 'use',
    'F29': 'storage',
    'F30': 'organic',
    'F31': 'vitamin-mineral-enrichment',
    'F32': 'labelling',
    'F33': 'flavour',
    'F34': 'ingredient',
}



In [15]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3.  PARSE allFacets STRING
# ═══════════════════════════════════════════════════════════════════════════════
# Format: "F01.A059P$F02.A066Q$F27.A000A$F33.A0C4A"
# The base term code (before #) is NOT in allFacets — it's already termCode.
# Each facet entry is FXX.CODE, separated by $.

FACET_PATTERN = re.compile(r'(F\d{2})\.([A-Z0-9]{4,6})')


def parse_allFacets(facets_str: str) -> list[dict]:
    """Parse allFacets → list of {facet_cat, facet_label, code, name}."""
    if not isinstance(facets_str, str) or not facets_str.strip():
        return []

    results = []
    for match in FACET_PATTERN.finditer(facets_str):
        cat  = match.group(1)           # e.g. "F01"
        code = match.group(2)           # e.g. "A059P"
        name = code_to_name.get(code)   # resolve via MTX lookup
        results.append({
            'facet_cat'   : cat,
            'facet_label' : FACET_LABELS.get(cat, cat),
            'code'        : code,
            'name'        : name,        # None if code not found in MTX
        })
    return results


# Quick sanity check
sample_facets = food_df['allFacets'].dropna().iloc[0]
print(f'\nSample allFacets: {sample_facets}')
print(f'Parsed         : {parse_allFacets(sample_facets)}')


Sample allFacets: A000A#F01.A059P$F02.A066Q$F27.A000A$F33.A0C4A
Parsed         : [{'facet_cat': 'F01', 'facet_label': 'source', 'code': 'A059P', 'name': 'Teff (as plant)'}, {'facet_cat': 'F02', 'facet_label': 'part-nature', 'code': 'A066Q', 'name': 'Grains (as part-nature)'}, {'facet_cat': 'F27', 'facet_label': 'geographical-origin', 'code': 'A000A', 'name': 'Teff grain'}, {'facet_cat': 'F33', 'facet_label': 'flavour', 'code': 'A0C4A', 'name': 'FA-06.1 Whole, broken, or flaked grain'}]


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 7.  SAVE
# ═══════════════════════════════════════════════════════════════════════════════
# Derive hierarchy depth from the dot-separated reportHierarchyCode
food_df['reportHierarchyLevel'] = food_df['reportHierarchyCode'].apply(
    lambda x: len(str(x).split('.')) if pd.notna(x) else None
)

# Drop feed-only terms (not relevant for human food matching)
mask_feed = food_df['termExtendedName'].str.contains(r'\(feed\)', case=False, na=False)
print(f'Dropping {mask_feed.sum():,} rows with "(feed)" in termExtendedName')
food_df = food_df[~mask_feed].reset_index(drop=True)

KEEP_COLS = [
    'termCode', 'termExtendedName', 'termScopeNote',
    'scientificNames', 'allFacets', 'reportHierarchyLevel',
    'nlp_matching_friendly',
]
food_df[KEEP_COLS].to_csv(OUTPUT_CSV, index=False)
print(f'\nSaved: {OUTPUT_CSV}')
print(f'Columns: {KEEP_COLS}')
print(f'Shape: {food_df[KEEP_COLS].shape}')

In [21]:
food_df.columns

Index(['termCode', 'termExtendedName', 'termShortName', 'termScopeNote',
       'version', 'lastUpdate', 'validFrom', 'validTo', 'status', 'deprecated',
       ...
       'PRIMoReportable', 'PRIMoHierarchyCode', 'hostsampledFlag',
       'hostsampledParentCode', 'hostsampledOrder', 'hostsampledReportable',
       'hostsampledHierarchyCode', 'facets_resolved', 'termExtendedName_full',
       'nlp_matching_friendly'],
      dtype='object', length=230)

In [25]:
# ═══════════════════════════════════════════════════════════════════════════════
# 7.  SAVE
# ═══════════════════════════════════════════════════════════════════════════════
food_df.to_csv(OUTPUT_CSV, index=False)
print(f'\nSaved: {OUTPUT_CSV}')
print(f'New columns: facets_resolved, termExtendedName_full')
print(f'Shape: {food_df.shape}')


Saved: /home/alexl/TFM/data/foodHierarchy_nlp_extended.csv
New columns: facets_resolved, termExtendedName_full
Shape: (5362, 230)
